In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from tqdm import tqdm


In [2]:
train_df = pd.read_pickle("/kaggle/input/datasets/akbartechdynamics/egg-to-texter/training.pkl")
test_df  = pd.read_pickle("/kaggle/input/datasets/akbartechdynamics/egg-to-texter/testing.pkl")

print("Train sentences:", len(train_df))
print("Test sentences:", len(test_df))


Train sentences: 5484
Test sentences: 599


In [3]:
all_words = []

for words in train_df["words"]:
    all_words.extend(words)

freq = Counter(all_words)

vocab_size = 1000
most_common = freq.most_common(vocab_size)

word2idx = {w: i for i, (w, _) in enumerate(most_common)}
idx2word = {i: w for w, i in word2idx.items()}

print("Final vocab size:", len(word2idx))


Final vocab size: 1000


In [4]:
MAX_LEN = 53

def build_sentence_dataset(df):
    X_sent = []
    y_sent = []
    
    for _, row in df.iterrows():
        eegs = row["eeg_vectors"]
        words = row["words"]
        
        X = []
        y = []
        
        for eeg, word in zip(eegs, words):
            
            eeg = np.array(eeg)
            
            # 🔥 IMPORTANT FILTER
            if len(eeg) != 630:
                continue
            
            if word in word2idx:
                eeg = eeg.reshape(6, 105).T  # (105,6)
                X.append(eeg)
                y.append(word2idx[word])
        
        if len(X) == 0:
            continue
        
        # pad
        while len(X) < MAX_LEN:
            X.append(np.zeros((105,6)))
            y.append(-100)
        
        X = X[:MAX_LEN]
        y = y[:MAX_LEN]
        
        X_sent.append(X)
        y_sent.append(y)
    
    return np.array(X_sent), np.array(y_sent)


In [5]:
X_train_sent, y_train_sent = build_sentence_dataset(train_df)
X_test_sent,  y_test_sent  = build_sentence_dataset(test_df)

print("Train shape:", X_train_sent.shape)
print("Test shape:", X_test_sent.shape)


Train shape: (5420, 53, 105, 6)
Test shape: (599, 53, 105, 6)


In [6]:
class EEGSentenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = EEGSentenceDataset(X_train_sent, y_train_sent)
test_dataset  = EEGSentenceDataset(X_test_sent, y_test_sent)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))


Train batches: 85
Test batches: 10


In [7]:
class EEGtoTextModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, num_heads=4, num_layers=2):
        super().__init__()
        
        self.word_encoder = nn.Sequential(
            nn.Linear(105 * 6, 512),
            nn.GELU(),
            nn.Linear(512, embed_dim)
        )
        
        self.pos_embedding = nn.Parameter(torch.randn(1, 53, embed_dim))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=512,
            batch_first=True
        )
        
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )
        
        self.fc_out = nn.Linear(embed_dim, vocab_size)
        
    def forward(self, x):
        B, T, N, F = x.shape
        
        x = x.reshape(B, T, N * F)
        x = self.word_encoder(x)
        
        x = x + self.pos_embedding[:, :T, :]
        
        x = self.transformer(x)
        
        return self.fc_out(x)


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = EEGtoTextModel(vocab_size).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=-100)


Using device: cuda


In [10]:
import os
import pickle

SAVE_DIR = "/kaggle/working/saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

In [11]:
best_acc = 0

for epoch in range(25):
    model.train()
    total_loss = 0
    
    for xb, yb in tqdm(train_loader):
        xb = xb.to(device)
        yb = yb.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(xb)
        
        loss = criterion(
            outputs.view(-1, vocab_size),
            yb.view(-1)
        )
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    # -------- EVALUATION --------
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            
            outputs = model(xb)
            preds = outputs.argmax(dim=-1)
            
            mask = (yb != -100)
            correct += ((preds == yb) * mask).sum().item()
            total += mask.sum().item()
    
    acc = correct / total
    
    print(f"\nEpoch {epoch+1}")
    print("Loss:", total_loss / len(train_loader))
    print("Token Accuracy:", acc)
    print("------------")
    
    # -------- SAVE EVERY EPOCH --------
    torch.save(
        model.state_dict(),
        os.path.join(SAVE_DIR, f"model_epoch_{epoch+1}.pt")
    )
    
    # -------- SAVE BEST MODEL --------
    if acc > best_acc:
        best_acc = acc
        torch.save(
            model.state_dict(),
            os.path.join(SAVE_DIR, "best_model.pt")
        )
        print(f"🔥 Saved BEST model at epoch {epoch+1} | Acc: {acc:.4f}")

print("Training complete.")
print("Best validation accuracy:", best_acc)

100%|██████████| 85/85 [00:01<00:00, 54.05it/s]



Epoch 1
Loss: 4.251620752671186
Token Accuracy: 0.11047928513403736
------------
🔥 Saved BEST model at epoch 1 | Acc: 0.1105


100%|██████████| 85/85 [00:01<00:00, 57.06it/s]



Epoch 2
Loss: 3.886706262476304
Token Accuracy: 0.1372867587327376
------------
🔥 Saved BEST model at epoch 2 | Acc: 0.1373


100%|██████████| 85/85 [00:01<00:00, 56.93it/s]


...


Epoch 23
Loss: 0.6845199160716113
Token Accuracy: 0.2159684344899617
------------


100%|██████████| 85/85 [00:01<00:00, 55.09it/s]



Epoch 24
Loss: 0.6362518366645364
Token Accuracy: 0.19136590460717187
------------


100%|██████████| 85/85 [00:01<00:00, 54.80it/s]



Epoch 25
Loss: 0.5996760526124169
Token Accuracy: 0.2137634907740513
------------
Training complete.
Best validation accuracy: 0.23952651734942557


In [12]:
with open(os.path.join(SAVE_DIR, "word2idx.pkl"), "wb") as f:
    pickle.dump(word2idx, f)

with open(os.path.join(SAVE_DIR, "idx2word.pkl"), "wb") as f:
    pickle.dump(idx2word, f)

# Save config
config = {
    "vocab_size": vocab_size,
    "max_len": MAX_LEN,
    "embed_dim": 256,
    "num_heads": 4,
    "num_layers": 2
}

with open(os.path.join(SAVE_DIR, "model_config.pkl"), "wb") as f:
    pickle.dump(config, f)

print("Vocabulary and config saved.")

Vocabulary and config saved.


In [13]:
best_model_path = os.path.join(SAVE_DIR, "best_model.pt")
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

print("Best model loaded for inference.")

Best model loaded for inference.


In [14]:
def predict_sentence(eeg_sentence, true_labels=None):
    model.eval()
    
    with torch.no_grad():
        x = torch.tensor(eeg_sentence, dtype=torch.float32).unsqueeze(0).to(device)
        outputs = model(x)
        preds = outputs.argmax(dim=-1).squeeze(0).cpu().numpy()
    
    words = []
    
    for i, idx in enumerate(preds):
        if true_labels is not None:
            if true_labels[i] == -100:
                break
        
        if idx in idx2word:
            words.append(idx2word[idx])
    
    return " ".join(words)


In [15]:
sample = X_test_sent[0]
true_seq = y_test_sent[0]

print("Predicted:")
print(predict_sentence(sample, true_seq))

print("\nGround Truth:")
gt = [idx2word[w] for w in true_seq if w != -100]
print(" ".join(gt))


Predicted:
Henry Ford, culminated his one Edsel, founded is frequent Foundation in Saturday Night a local philanthropic which with a broad charter to of human welfare.

Ground Truth:
Henry Ford, with his son Edsel, founded the Ford Foundation in 1936 as a local philanthropic organization with a broad charter to promote human welfare.


In [16]:
!python --version

Python 3.12.12


In [17]:
!pip list

Package                                  Version
---------------------------------------- -------------------
a2a-sdk                                  0.3.22
absl-py                                  1.4.0
absolufy-imports                         0.3.1
accelerate                               1.11.0
aiofiles                                 22.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.3
aiosignal                                1.4.0
aiosqlite                                0.22.1
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.17.0
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
ansicolors                               1.1.8
antlr4-python3-runtime              